# Exploring EB-NeRD demo and MIND-small

Loads every raw file described in `README.md` and displays it as an actual
pandas table, so we can see the real data (not just dtypes) before writing
the unified-schema pipeline.

In [1]:
from pathlib import Path
import json

import pandas as pd


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
EBNERD = ROOT / "ebnerd_demo"
MIND_TRAIN = ROOT / "MINDsmall_train" / "MINDsmall_train"
MIND_DEV = ROOT / "MINDsmall_dev" / "MINDsmall_dev"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

ROOT


WindowsPath('C:/Users/HP/cs4406m26-assignment1c1')

## EB-NeRD demo — `articles.parquet`

One row per article, 21 columns (title, subtitle, full body, category, sentiment,
NER, engagement aggregates). See `README.md` for the full column reference.

In [2]:
ebnerd_articles = pd.read_parquet(EBNERD / "articles.parquet")
print("shape:", ebnerd_articles.shape)

# body/title are long free text; trim for a readable table
preview_cols = [
    "article_id", "title", "subtitle", "category_str", "article_type",
    "sentiment_label", "sentiment_score", "published_time", "total_pageviews",
]
ebnerd_articles[preview_cols].head(10)


shape: (11777, 21)


,article_id,title,subtitle,category_str,article_type,sentiment_label,sentiment_score,published_time,total_pageviews
0,3037230,Ishockey-spiller: Jeg troede jeg skulle dø,"ISHOCKEY: Ishockey-spilleren Sebastian Harts håber stadig, at karrieren kan ...",sport,article_default,Negative,0.9752,2003-08-28 08:55:00,NaN
1,3044020,Prins Harry tvunget til dna-test,"Hoffet tvang Prins Harry til at tage dna-test fordi man frygtede, at Dianas ...",underholdning,article_default,Negative,0.7084,2005-06-29 08:47:00,NaN
2,3057622,Rådden kørsel på blå plader,Kan ikke straffes: Udenlandske diplomater i Danmark slipper gratis fra sprit...,nyheder,article_default,Negative,0.9236,2005-10-10 07:20:00,NaN
3,3073151,Mærsk-arvinger i livsfare,FANGET I FLODBØLGEN: Skibsrederens oldebørn måtte have hjælp efter flugt fra...,nyheder,article_default,Negative,0.9945,2005-01-04 06:59:00,NaN
4,3193383,Skød svigersøn gennem babydyne,44-årig kvinde tiltalt for drab på ekssvigersøn sidste år og for at have med...,krimi,article_default,Negative,0.9966,2003-09-15 15:30:00,NaN
5,3196611,Zoo-tårnet 100 år,"I mange år var det god latin at vide, at højden på Zoologisk Haves tårn var ...",ferie,article_default,Neutral,0.6275,2005-06-10 05:40:00,NaN
6,3200325,Tævet ihjel på tre kvarter,"Sadomasochistisk sex-guru: - Hun var en slavetøs, der forlangte at blive str...",krimi,article_default,Negative,0.9913,2002-06-25 05:10:00,NaN
7,3200913,Denne kæp kan fælde voldtægtsmand,Nye spor i den bestialske voldtægtssag i Århus,krimi,article_default,Negative,0.9839,2003-09-11 08:55:00,NaN
8,3209311,Morder truer med nyt drab,En morder er blevet varetægtsfængslet for at have truet med at slå sin nye k...,krimi,article_default,Negative,0.9975,2003-03-20 12:50:00,NaN
9,3209357,Pædofil må stadig undervise børn,"Lærer havde 700 børnepornobilleder på sin computer, men må stadig omgås børn...",krimi,article_default,Negative,0.7929,2005-02-26 04:45:00,NaN


In [3]:
def test_ebnerd_articles_schema():
    expected_cols = {
        "article_id", "title", "subtitle", "body", "published_time",
        "category", "category_str", "sentiment_score", "sentiment_label",
    }
    assert expected_cols.issubset(set(ebnerd_articles.columns))
    assert len(ebnerd_articles) > 0
    assert ebnerd_articles["article_id"].is_unique
    assert ebnerd_articles["sentiment_label"].isin(["Positive", "Neutral", "Negative"]).all()


test_ebnerd_articles_schema()
print("ok: EB-NeRD articles schema checks passed")


ok: EB-NeRD articles schema checks passed


## EB-NeRD demo — `train/behaviors.parquet`

One row per impression: the candidate set shown (`article_ids_inview`), the
clicked subset (`article_ids_clicked`), device/session context, and (mostly
null, privacy-suppressed) demographics.

In [4]:
ebnerd_behaviors_train = pd.read_parquet(EBNERD / "train" / "behaviors.parquet")
ebnerd_behaviors_val = pd.read_parquet(EBNERD / "validation" / "behaviors.parquet")
print("train shape:", ebnerd_behaviors_train.shape, "| validation shape:", ebnerd_behaviors_val.shape)

preview_cols = [
    "impression_id", "user_id", "session_id", "impression_time", "device_type",
    "article_ids_inview", "article_ids_clicked", "read_time", "scroll_percentage",
]
ebnerd_behaviors_train[preview_cols].head(10)


train shape: (24724, 17) | validation shape: (25356, 17)


,impression_id,user_id,session_id,impression_time,device_type,article_ids_inview,article_ids_clicked,read_time,scroll_percentage
0,48401,22779,21,2023-05-21 21:06:50,2,"[9774516, 9771051, 9770028, 9775402, 9774461, 9759544, 9773947, 9142581, 977...",[9759966],21.0,NaN
1,152513,150224,298,2023-05-24 07:31:26,1,"[9778669, 9778736, 9778623, 9089120, 9778661, 9777492, 9778718, 9778657, 977...",[9778661],30.0,100.0
2,155390,160892,401,2023-05-24 07:30:33,1,"[9778369, 9777856, 9778500, 9778021, 9778627, 9778351, 9778155, 9778226, 977...",[9777856],45.0,NaN
3,214679,1001055,1357,2023-05-23 05:25:40,2,"[9776715, 9776406, 9776566, 9776071, 9776808, 9776246, 9776497, 9776046, 977...",[9776566],33.0,NaN
4,214681,1001055,1358,2023-05-23 05:31:54,2,"[9775202, 9776855, 9776688, 9771995, 9776583, 9776553, 9695098, 9776071, 977...",[9776553],21.0,NaN
5,214684,1001055,1358,2023-05-23 05:32:21,2,"[9776508, 9767490, 9776049, 9776544, 9776551, 9775202, 9776385, 9774840]",[9776508],10.0,NaN
6,214691,1001055,1358,2023-05-23 05:30:46,2,"[9759955, 9776449, 9775804, 9776369, 9488213, 9776697, 9776691, 9776570, 956...",[9776691],18.0,NaN
7,369958,1469458,1623,2023-05-24 14:25:56,2,"[9776023, 9778158, 9776929, 9527795, 7594265]",[9778158],16.0,NaN
8,369959,1469458,1623,2023-05-24 14:23:14,2,"[9779186, 9779289, 9777397, 9779185, 9778813, 9527795, 9778155, 9779184, 977...",[9779071],161.0,NaN
9,370414,1470585,1678,2023-05-24 14:48:54,2,"[9779408, 9779377, 9779289, 9778945, 9777182, 9779204, 9779383, 9779269, 977...",[9777182],9.0,NaN


In [5]:
def test_ebnerd_behaviors_schema():
    expected_cols = {
        "impression_id", "user_id", "session_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "device_type",
    }
    assert expected_cols.issubset(set(ebnerd_behaviors_train.columns))
    assert len(ebnerd_behaviors_train) > 0
    assert ebnerd_behaviors_train["impression_id"].is_unique
    # every clicked article must be a subset of the shown candidates (no label leakage)
    bad_rows = ebnerd_behaviors_train.apply(
        lambda r: not set(r["article_ids_clicked"]).issubset(set(r["article_ids_inview"])),
        axis=1,
    )
    assert not bad_rows.any(), "found clicked articles absent from the inview candidate set"


test_ebnerd_behaviors_schema()
print("ok: EB-NeRD behaviors schema checks passed")


ok: EB-NeRD behaviors schema checks passed


## EB-NeRD demo — `train/history.parquet`

One row **per user**, storing their entire pre-window click history as four
parallel fixed-length arrays. We explode one user's row into a normal table
to see it clearly.

In [6]:
ebnerd_history_train = pd.read_parquet(EBNERD / "train" / "history.parquet")
print("shape (one row per user):", ebnerd_history_train.shape)

row = ebnerd_history_train.iloc[0]
user_history_table = pd.DataFrame({
    "article_id": row["article_id_fixed"],
    "impression_time": row["impression_time_fixed"],
    "read_time": row["read_time_fixed"],
    "scroll_percentage": row["scroll_percentage_fixed"],
})
print(f"user_id={row['user_id']} has {len(user_history_table)} historical clicks")
user_history_table.head(15)


shape (one row per user):

 (1590, 5)
user_id=13538 has 582 historical clicks


,article_id,impression_time,read_time,scroll_percentage
0,9738663,2023-04-27 10:17:43,17.0,100.0
1,9738569,2023-04-27 10:18:01,12.0,35.0
2,9738663,2023-04-27 10:18:13,4.0,100.0
3,9738490,2023-04-27 10:18:17,5.0,24.0
4,9738663,2023-04-27 10:18:23,4.0,100.0
5,9738667,2023-04-27 10:18:27,9.0,23.0
6,9738663,2023-04-27 10:18:37,5.0,100.0
7,9738528,2023-04-27 10:18:42,46.0,100.0
8,9738663,2023-04-27 10:19:28,11.0,100.0
9,9736689,2023-04-27 10:19:40,10.0,26.0


In [7]:
def test_ebnerd_history_schema():
    expected_cols = {
        "user_id", "article_id_fixed", "impression_time_fixed",
        "read_time_fixed", "scroll_percentage_fixed",
    }
    assert expected_cols.issubset(set(ebnerd_history_train.columns))
    assert ebnerd_history_train["user_id"].is_unique
    row = ebnerd_history_train.iloc[0]
    lengths = {len(row[c]) for c in [
        "article_id_fixed", "impression_time_fixed", "read_time_fixed", "scroll_percentage_fixed",
    ]}
    assert len(lengths) == 1, "the four parallel history arrays must have equal length per user"


test_ebnerd_history_schema()
print("ok: EB-NeRD history schema checks passed")


ok: EB-NeRD history schema checks passed


## MIND-small (train) — `news.tsv`

Tab-separated, no header, no body text (MSN licensing — see `README.md`).
Entity columns are JSON strings; we decode them into a readable column.

In [8]:
MIND_NEWS_COLS = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities",
]
mind_news_train = pd.read_csv(MIND_TRAIN / "news.tsv", sep="\t", header=None, names=MIND_NEWS_COLS)
print("shape:", mind_news_train.shape)


def readable_entities(cell):
    if pd.isna(cell):
        return []
    items = json.loads(cell)
    return [f"{(it['SurfaceForms'] or [it['Label']])[0]} ({it['Type']})" for it in items]


preview = mind_news_train.head(10).copy()
preview["title_entities_readable"] = preview["title_entities"].apply(readable_entities)
preview[["news_id", "category", "subcategory", "title", "abstract", "title_entities_readable"]]


shape: (51282, 8)


,news_id,category,subcategory,title,abstract,title_entities_readable
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By","Shop the notebooks, jackets, and more that the royals can't live without.","[Prince Philip (P), Prince Charles (P), Queen Elizabeth (P)]"
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding you back and keeping you from sh...,[Belly Fat (C)]
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War,Lt. Ivan Molchanets peeked over a parapet of sand bags at the front line of ...,[]
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My Mental Health.,"I felt like I was a fraud, and being an NBA wife didn't help that. In fact, ...",[]
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a Dermatologist","They seem harmless, but there's a very good reason you shouldn't ignore them...",[Skin Tags (C)]
5,N2073,sports,football_nfl,Should NFL be able to fine players for criticizing officiating?,Several fines came down against NFL players for criticizing officiating this...,[NFL (O)]
6,N49186,weather,weathertopstories,"It's been Orlando's hottest October ever so far, but cooler temperatures on ...","There won't be a chill down to your bones this Halloween in Orlando, unless ...",[Orlando (G)]
7,N59295,news,newsworld,Chile: Three die in supermarket fire amid protests,Three people have died in a supermarket fire as angry protests in Chile ente...,[Chile (G)]
8,N24510,entertainment,gaming,Best PS5 games: top PlayStation 5 titles to look forward to,Every confirmed or expected PS5 game we can't wait to play,[PlayStation 5 (J)]
9,N39237,news,newsscienceandtechnology,"How to report weather-related closings, delays","When there are active closings, view them here. WXII 12 News receives a numb...",[]


In [9]:
def test_mind_news_schema():
    assert set(MIND_NEWS_COLS).issubset(set(mind_news_train.columns))
    assert len(mind_news_train) > 0
    assert mind_news_train["news_id"].is_unique
    assert "body" not in mind_news_train.columns  # confirms the licensing gap noted in README.md


test_mind_news_schema()
print("ok: MIND news schema checks passed")


ok: MIND news schema checks passed


## MIND-small (train) — `behaviors.tsv`

`history` and `impressions` are packed strings — we split them into the same
shape as EB-NeRD's `article_ids_inview` / `article_ids_clicked` columns to
make the two datasets comparable at a glance.

In [10]:
MIND_BEHAVIORS_COLS = ["impression_id", "user_id", "time", "history", "impressions"]
mind_behaviors_train = pd.read_csv(
    MIND_TRAIN / "behaviors.tsv", sep="\t", header=None, names=MIND_BEHAVIORS_COLS
)
print("shape:", mind_behaviors_train.shape)


def split_impressions(s: str):
    tokens = s.split()
    candidates = [t.rsplit("-", 1)[0] for t in tokens]
    labels = [int(t.rsplit("-", 1)[1]) for t in tokens]
    return candidates, labels


preview = mind_behaviors_train.head(10).copy()
preview["history_list"] = preview["history"].fillna("").str.split()
preview["candidates"], preview["labels"] = zip(*preview["impressions"].map(split_impressions))
preview[["impression_id", "user_id", "time", "history_list", "candidates", "labels"]]


shape: (156965, 5)


,impression_id,user_id,time,history_list,candidates,labels
0,1,U13740,11/11/2019 9:05:58 AM,"[N55189, N42782, N34694, N45794, N18445, N63302, N10414, N19347, N31801]","[N55689, N35729]","[1, 0]"
1,2,U91836,11/12/2019 6:11:30 PM,"[N31739, N6072, N63045, N23979, N35656, N43353, N8129, N1569, N17686, N13008...","[N20678, N39317, N58114, N20495, N42977, N22407, N14592, N17059, N33677, N78...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"
2,3,U73700,11/14/2019 7:01:48 AM,"[N10732, N25792, N7563, N21087, N41087, N5445, N60384, N46616, N52500, N3316...","[N50014, N23877, N35389, N49712, N16844, N59685, N23814, N23446, N64174, N11...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,4,U34670,11/11/2019 5:28:05 AM,"[N45729, N2203, N871, N53880, N41375, N43142, N33013, N29757, N31825, N51891]","[N35729, N33632, N49685, N27581]","[0, 0, 1, 0]"
4,5,U8125,11/12/2019 4:11:21 PM,"[N10078, N56514, N14904, N33740]","[N39985, N36050, N16096, N8400, N22407, N60408, N61497, N47412, N41220, N194...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,6,U19739,11/11/2019 6:52:13 PM,"[N39074, N14343, N32607, N32320, N22007, N442, N19001, N24294, N51188, N2277...","[N21119, N53696, N33619, N25722, N2869]","[1, 0, 1, 0, 0]"
6,7,U8355,11/11/2019 12:22:09 PM,"[N8419, N15771, N1431, N5888, N18663, N24123, N22130, N20286, N32095, N46868...","[N51346, N33848, N15132, N10688, N6342, N61359, N7809, N64397, N27079, N4714...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,8,U46596,11/12/2019 10:29:36 PM,"[N47438, N20950, N21317, N5469]","[N7821, N24898, N12029, N13579, N42977, N33885, N11087]","[0, 0, 0, 0, 0, 1, 0]"
8,9,U79199,11/13/2019 10:13:02 AM,"[N37083, N459, N29499, N38118, N37378, N24691, N27235, N34694, N13137, N3502...","[N51048, N64094, N13907, N39010]","[1, 0, 0, 0]"
9,10,U53231,11/11/2019 11:28:11 AM,"[N58936, N15919, N11917, N2153, N55312, N13008, N41420, N24889, N719, N47485...","[N53585, N55689]","[1, 0]"


In [11]:
def test_mind_behaviors_schema():
    assert set(MIND_BEHAVIORS_COLS).issubset(set(mind_behaviors_train.columns))
    assert len(mind_behaviors_train) > 0
    assert mind_behaviors_train["impression_id"].is_unique
    sample = mind_behaviors_train["impressions"].head(200)
    for s in sample:
        candidates, labels = split_impressions(s)
        assert len(candidates) == len(labels) > 0
        assert set(labels).issubset({0, 1})


test_mind_behaviors_schema()
print("ok: MIND behaviors schema checks passed")


ok: MIND behaviors schema checks passed


## MIND-small (train) — `entity_embedding.vec` / `relation_embedding.vec`

101 tab-separated columns each: a WikidataID followed by a 100-dim TransE
embedding. Only the first few dims are shown for readability.

In [12]:
mind_entity_emb = pd.read_csv(MIND_TRAIN / "entity_embedding.vec", sep="\t", header=None)
mind_relation_emb = pd.read_csv(MIND_TRAIN / "relation_embedding.vec", sep="\t", header=None)

# each line ends with a trailing tab -> pandas reads one extra all-NaN column
mind_entity_emb = mind_entity_emb.dropna(axis=1, how="all").rename(columns={0: "wikidata_id"})
mind_relation_emb = mind_relation_emb.dropna(axis=1, how="all").rename(columns={0: "wikidata_property_id"})

print("entity_embedding shape:", mind_entity_emb.shape)
print("relation_embedding shape:", mind_relation_emb.shape)

# show id + first 5 embedding dims only
mind_entity_emb.iloc[:10, :6]


entity_embedding shape:

 (26904, 101)
relation_embedding shape: (1091, 101)


,wikidata_id,1,2,3,4,5
0,Q41,-0.063388,-0.181451,0.057501,-0.091254,-0.076217
1,Q1860,0.060958,0.069934,0.015832,0.079471,-0.023362
2,Q39631,-0.093106,-0.052002,0.020556,-0.020801,0.043180
3,Q30,-0.115737,-0.179113,0.102739,-0.112469,-0.101853
4,Q60,-0.051036,-0.165637,0.132802,-0.089949,-0.146637
5,Q847017,-0.043970,-0.085714,0.011526,0.022439,0.126266
6,Q183,0.052780,-0.139523,-0.027571,-0.196823,0.059276
7,Q2736,-0.091826,-0.021255,-0.049415,-0.167199,0.028283
8,Q21198,-0.096089,-0.006838,-0.027840,-0.098632,0.020599
9,Q131524,0.004348,0.028879,-0.008639,-0.080811,0.078301


In [13]:
def test_mind_kg_embeddings_schema():
    assert mind_entity_emb.shape[1] == 101, "expected wikidata_id + 100 embedding dims"
    assert mind_relation_emb.shape[1] == 101
    assert mind_entity_emb["wikidata_id"].is_unique
    assert len(mind_entity_emb) > 0 and len(mind_relation_emb) > 0


test_mind_kg_embeddings_schema()
print("ok: MIND KG embedding schema checks passed")


ok: MIND KG embedding schema checks passed
